In [200]:
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool, OpenAIChatCompletionsModel
from agents.model_settings import ModelSettings
from pydantic import BaseModel
from dotenv import load_dotenv
from openai import AsyncOpenAI
from serpapi.google_search import GoogleSearch
import re
import time
import requests
import asyncio
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup
from dateutil import parser as dateparser
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict
from IPython.display import display, Markdown
from typing import Optional, List

In [201]:
load_dotenv(override=True)

True

In [202]:
SERPAPI_KEY = os.getenv("SERPAPI_API_KEY")

In [203]:
openai_api_key = os.getenv('OPENAI_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")
if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AI
Groq API Key exists and begins gsk_


In [204]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"


In [205]:
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)
llama3_3_model = OpenAIChatCompletionsModel(model="llama-3.3-70b-versatile", openai_client=groq_client)

In [206]:
HEADERS = {"User-Agent": "FundingScraper/1.0 (research; contact: you@example.com)"}
KEYWORDS = re.compile(r"(ph\.?d|doctoral|doctorate|masters?|m\.sc|mres|scholarship|funding|studentship|stipend)", re.I)


In [207]:
@function_tool
def find_university_domain(school: str, country: Optional[str]=None, num: int=5) -> List[str]:
    """
    Find likely university domains for any school in any country using SerpAPI.
    - school: "University of Melbourne"
    - country: "Australia" (optional)
    Returns: list of domains (https://...)
    """

    query_parts = [school, "official site", "scholarship", "funding"]
    if country:
        query_parts.append(country)
    query = " ".join(query_parts)

    search = GoogleSearch({
        "q": query,
        "api_key": SERPAPI_KEY,
        "num": num
    })
    results = search.get_dict()
    print(results)

    urls = []
    if "organic_results" in results:
        for r in results["organic_results"]:
            if "link" in r:
                urls.append(r["link"])

    # Filter to probable university domains
    cleaned = []
    for u in urls:
        netloc = urlparse(u).netloc.lower()
        if any(x in netloc for x in [school.lower().replace(" ", ""), "univ", "edu", "ac."]):
            base = f"https://{netloc}"
            if base not in cleaned:
                cleaned.append(base)

    return cleaned[:num]

In [ ]:
class SchoolAndDomain(BaseModel):
    school: str
    "The name of the school"
    domain: str
    "The school's official domain"

In [209]:
search_agent_instructions = """You are a university domain research assistant. 

FOLLOW THESE RULES:
1. FIRST, check if specific school names are EXPLICITLY mentioned in the query
2. IF explicit schools are mentioned:
   - Return ONLY those explicitly mentioned schools and their domains
   - Do NOT add any other schools
3. IF NO explicit schools are mentioned:
   - Search for relevant schools based on the query context
   - Return the most relevant schools and their domains

Examples:
- "Find MIT and Stanford domains" → Return: MIT, Stanford (only explicit)
- "University of Toronto website" → Return: University of Toronto (only explicit)
- "Top AI PhD programs" → Return: List of relevant schools (no explicit ones)
- "Machine learning scholarships in UK" → Return: List of relevant UK schools"""

class MultipleSchoolsAndDomains(BaseModel):
    schools: List[SchoolAndDomain]
    "List of schools and their domains"
    search_type: str
    "Either 'explicit' (schools were explicitly mentioned) or 'searched' (schools were found via search)"

search_agent = Agent(
    name="Smart university domain finder",
    instructions=search_agent_instructions,
    tools=[find_university_domain],
    model="gpt-4o-mini",
    output_type=MultipleSchoolsAndDomains
)



In [210]:
def remove_duplicate_schools(schools_list: List[SchoolAndDomain]) -> List[SchoolAndDomain]:
    """Remove duplicate schools from the list"""
    seen = set()
    unique_schools = []
    
    for school_domain in schools_list:
        # Normalize school name for comparison
        normalized_name = school_domain.school.lower().strip()
        
        if normalized_name not in seen:
            seen.add(normalized_name)
            unique_schools.append(school_domain)
        else:
            print(f"Skipped duplicate: {school_domain.school}")
    
    return unique_schools

In [213]:
if __name__ == "__main__":
    school = " top 5 Phd machine learning funding"
    country = "United Kingdom"  # change this to any country
    query = f"Find domain for {school} in {country}"
    with trace("Search"):
        print("searching...")
        print(f"searching: {query}")
        search = await Runner.run(search_agent,query)
        all_schools = []

        all_schools.extend(search.final_output.schools)
    
    #schools = find_university_domain(school,country)
    unique_schools = remove_duplicate_schools(all_schools)
    print(all_schools)
    #print(result)

searching...
searching: Find domain for  top 5 Phd machine learning funding in United Kingdom
{'search_metadata': {'id': '68de1d7348c7915816bcc5b7', 'status': 'Success', 'json_endpoint': 'https://serpapi.com/searches/234ed44791d54d2c/68de1d7348c7915816bcc5b7.json', 'pixel_position_endpoint': 'https://serpapi.com/searches/234ed44791d54d2c/68de1d7348c7915816bcc5b7.json_with_pixel_position', 'created_at': '2025-10-02 06:36:35 UTC', 'processed_at': '2025-10-02 06:36:35 UTC', 'google_url': 'https://www.google.com/search?q=University+of+Cambridge+official+site+scholarship+funding+United+Kingdom&oq=University+of+Cambridge+official+site+scholarship+funding+United+Kingdom&num=5&sourceid=chrome&ie=UTF-8', 'raw_html_file': 'https://serpapi.com/searches/234ed44791d54d2c/68de1d7348c7915816bcc5b7.html', 'total_time_taken': 1.03}, 'search_parameters': {'engine': 'google', 'q': 'University of Cambridge official site scholarship funding United Kingdom', 'google_domain': 'google.com', 'num': '5', 'devic